# Predictive Anayltics: Support Vector Machines with Regression

Task - Approach for SVM:
- Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
- How good is your model? Evaluate your model’s performance and comment on its shortfalls.
- Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
- How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

We used the GPU to train this model. In case the model shoulde be trained on the CPU. Change USE_GPU to false.

In [2]:
USE_GPU = False
from run_config import PATHS, MODELS_DIR

In [3]:
if USE_GPU:
    %load_ext cuml.accel
from run_config import PATHS

In [4]:
if USE_GPU:
    import os
    os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

    import cuml
    print(cuml.__version__)

In [ ]:
TRAIN_SAMPLE = 70_000
GRID_SAMPLE = 70_000 # for running on a subset to get good parameter and the use the whole set validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "HEXAGON" # HEXAGON
SPATIAL_ENCODING = "latlong" # options: embedding, latlong
MODE = "full" # options: full, sample
TIME_UNIT = "1H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [6]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [ ]:
if TIME_UNIT == "24H":
    N_COMP = [10, 30]
else:
    N_COMP = [100, 300]

In [7]:
import pandas as pd
import numpy as np
#import matplotlib as plt
import datetime
if USE_GPU:
    from cuml import SVR
    from cuml import LinearSVR
else:
    from sklearn.svm import SVR 
    from sklearn.svm import LinearSVR
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample
from sklearn.model_selection import GridSearchCV
#from sklearn.model_selection import RandomizedSearchCV 
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3
from joblib import Memory


## Preparations

In [8]:
INPUT = PATHS.train_test_dir

In [9]:
DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_TRAIN.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_TEST.parquet"

In [10]:
MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [11]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [12]:
if len(train_df) > TRAIN_SAMPLE:
   train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=42)

In [13]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
1149267,2025-02-21 21:00:00,2,5,21,0.500000,8.660254e-01,-0.433884,-0.900969,-0.707107,0.707107,...,0.0,36.6,0.522857,0.0,3.0,628.99,8.985571,4.75,15.25,Cash
883776,2025-05-11 16:00:00,5,7,16,0.866025,-5.000000e-01,-0.781831,0.623490,-0.866025,-0.500000,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
852395,2025-04-04 00:00:00,4,5,0,1.000000,6.123234e-17,-0.433884,-0.900969,0.000000,1.000000,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
977376,2025-09-19 14:00:00,9,5,14,-0.866025,-5.000000e-01,-0.433884,-0.900969,-0.500000,-0.866025,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
894581,2026-04-08 00:00:00,4,3,0,1.000000,6.123234e-17,0.974928,-0.222521,0.000000,1.000000,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips


In [14]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [15]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [16]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: hexa")
    for df in (train_df, val_df, test_df):
        df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

Encoding: latlong and Unit: hexa


In [17]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else: 
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

Create y

In [18]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Grid Search

In [19]:
model = SVR()

In [ ]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.001, 0.01, 0.1],
    "regressor__feature_map__n_components": N_COMP, # 24H: 10, 30; 1H/4H: 100, 300
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.001, 0.01, 0.1],
    "regressor__feature_map__n_components": N_COMP, # 24H: 10, 30; 1H/4H: 100, 300
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.0926843000462439 best params: {'regressor__svm__C': 1, 'regressor__svm__epsilon': 0.1}


C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficien

rbf_sigmoid best score: 0.19567512404299453 best params: {'regressor__feature_map__gamma': 0.001, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 30, 'regressor__svm__C': 1, 'regressor__svm__epsilon': 0.05}


C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficient evaluation of the full kernel.
  warnings.warn(
C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\kernel_approximation.py:1041: UserWarning: n_components > n_samples. This is not possible.
n_components was set to n_samples, which results in inefficien

poly best score: 0.21415465309384912 best params: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.1, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 30, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
Overall best: poly {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.1, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 30, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}


In [21]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.1, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 30, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
Best CV score: 0.21415465309384912


### Train Model

In [22]:
best_model = grid_search.best_estimator_

In [24]:
# Train SVR 
best_model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(memo..., tol=0.01))])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](41,)","['month_sin','month_cos','weekday_sin',...,'x','y','z']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,41
regressor_ regressor_: objectFitted regressor.,Pipeline,"Pipeline(memo..., tol=0.01))])"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,StandardScaler,StandardScaler()
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('feature_map', ...), ...]"


In [25]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [28]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


result = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2 Score": r2_score(y_test, y_pred),
}

MAE: 2.568545516591472
MSE: 223.74418276929583
RMSE: 14.958080851810363
R2 Score: 0.24341788788359142


In [30]:
df = pd.DataFrame({ # did not reorder at any point
    "y_pred": y_pred,
    "y_test": y_test,
    "h3_cell": test_df["h3_cell"].values,
    "date": test_df["datetime_hour"].values,
})
df.to_csv(MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}.csv", index=False)

In [32]:
pd.DataFrame([result]).to_csv(MODELS_DIR / f"svm/result_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}.csv", index=False)

In [33]:
dump(best_model, MODELS_DIR / f"svm/model_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}_svr.joblib")
dump(grid_search, MODELS_DIR / f"svm/grid_{SPATIAL_UNIT}_{H3_RES}_{TIME_UNIT}_svr.joblib")

['C:\\Users\\bkran\\Documents\\AAA\\Group-3-AAA\\models\\full\\svm\\grid_HEXAGON_1H_svr.joblib']